##### In LangChain, there are three primary ways to create custom tools depending on the complexity of your use case:   
* 1 @tool decorator (easiest & recommended for most functions)
* 2 StructuredTool with Pydantic (best when you need custom input schema validation)
* 3 Subclassing BaseTool (best for class-based state management or complex error handling)

##### Method 1 @tool decorator

+ When working with LLMs, agents, and API responses, malformed JSON string outputs are a constant headaches. This tool cleans raw text into formatted JSON and verifies its validity locally.

In [3]:
import json
from langchain_core.tools import tool

In [11]:
# step 1 - create a function
def multiply(a,b):
    """Multiply two numbers"""
    return a*b

In [12]:
# step 2 - add type hints
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [13]:
# step 3 - add tool decorator
@tool
def multiply(a: int,b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [15]:
result = multiply.invoke({"a":3,"b":5})
print(result)

15


In [16]:
print(multiply.description)
print(multiply.name)
print(multiply.args)

Multiply two numbers
multiply
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [17]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


#### Method 2 StructuredTool
* A Structured tools in langchain is a special type of tools where the input to the tool follows schema, typically defined using a pydantic model

In [3]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel,Field

In [4]:
class MultipltInput(BaseModel):
    a:int = Field(required = True, description="The first number to add")
    b:int = Field(required=True, description="The second number to add")

/var/folders/m5/6qc5jvtd7jq6xr1n9xf984rr0000gn/T/ipykernel_6193/3801108503.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int = Field(required = True, description="The first number to add")
/var/folders/m5/6qc5jvtd7jq6xr1n9xf984rr0000gn/T/ipykernel_6193/3801108503.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int = Field(required=True, description="The second number to add")


In [5]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [6]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name='multiply',
    description='multiply two numbers',
    args_schema=MultipltInput # Pydantic class
)

In [9]:
result = multiply_tool.invoke({'a':3, 'b':3})
print(result)

9


#### Method 3 - Using BaseTool Class

In [10]:
from langchain_core.tools import BaseTool
from typing import Type

In [11]:
# arg schema using pydantic
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "multiply two numbers"

    args_schema: Type[BaseModel] = MultipltInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [14]:
multiply_tool = MultiplyTool()

In [15]:
result = multiply_tool.invoke({'a':3, 'b':3})